# Lapisan 1 — Tahap 1 — **YOLOv12n, SATU lipatan, 30 epoch**

Lari verifikasi yang bisa dijalankan sendiri di satu GPU dalam ~20 menit, memakai
**setelan yang sama persis** dengan lari 3 lipatan yang angkanya masuk naskah
(`yolo12n.pt`, 30 epoch, imgsz 640, seed 42, arm `base`, `workers=0`).

Notebook ini **memanggil** fungsi di `y12.py` dan `centre_eval_folds.py`; ia tidak
menyalin ulang logikanya, supaya hasilnya tidak mungkin berbeda dari jalur skrip.

---

## ⚠ Baca ini sebelum memakai angkanya

**Satu lipatan, satu seed. Tidak ada mean ± std.**

Akibatnya, dan ini mengikat:

1. Aturan putusan `paired()` yang dipakai di seluruh paket ini **tidak berlaku** —
   ia butuh pasangan (lipatan, seed), dan di sini hanya ada satu.
2. Angka dari notebook ini **tidak boleh** diadu dengan **F1 pusat 0,960 ± 0,024**
   milik lari 3 lipatan. Itu perbandingan satu titik lawan sebuah rata-rata, persis
   kesalahan yang dijaga `00_ANGKA_FINAL.md` bagian E.
3. Ia **tidak boleh** dilaporkan sebagai replikasi, dan tidak boleh masuk
   `00_RINGKASAN.csv`.

Perlakuannya sama dengan jalur bukti Peru (`00_HASIL.md` §2.6): **dilaporkan
sendiri, dengan batasnya di depan.**

**Yang SAH dibaca dari notebook ini:** apakah pipeline Tahap 1 berjalan ujung ke
ujung di mesin ini, dan apakah lipatan yang dipilih mendarat di sekitar angka
yang tercatat untuk lipatan yang SAMA di lari 3 lipatan. Perbandingan
lipatan-lawan-lipatan-yang-sama itu sah; perbandingan terhadap rata-rata tidak.

Hasil ditulis ke ruang nama terpisah (`tag_suffix="1fold"`), jadi ia **tidak dapat
menimpa** `yolo12_results/yolo12n_base.json` maupun bobot 3 lipatan yang dipakai demo.


## 0 · Setelan

`FOLD` adalah satu-satunya hal yang biasanya perlu diubah.

| lipatan | ortomosaik yang DITAHAN | catatan |
|---|---|---|
| `fold0` | `44000_16000` | default |
| `fold1` | `44000_4000`   | |
| `fold2` | `52000_20000`  | anotasinya paling tidak konsisten (CV kotak 0,327) — lipatan tersulit |


In [1]:
import json
import os
import sys
import time

BASE = os.getcwd()                      # notebook dijalankan dari layer1_build/
if os.path.basename(BASE) != "layer1_build":
    cand = os.path.join(BASE, "layer1_build")
    if os.path.isdir(cand):
        os.chdir(cand); BASE = cand
    else:
        raise SystemExit("jalankan notebook ini dari dalam layer1_build/")
sys.path.insert(0, BASE)

FOLD        = "fold0"          # <- ubah di sini
MODEL       = "yolo12n.pt"
EPOCHS      = 30
IMGSZ       = 640
SEED        = 42
ARM         = "base"           # default Ultralytics apa adanya
TAG_SUFFIX  = "1fold"          # memisahkan ruang nama dari lari 3 lipatan
BATCH       = None             # None = 16 pada imgsz<=640; turunkan ke 8 bila OOM
CACHE       = "ram"            # "disk" bila RAM < 16 GB
WORKERS     = 0                # WAJIB 0 di notebook Windows (lihat y12.train_arm)

# Ambang keyakinan. TIDAK dipilih di sini - lihat bagian 3.
CONF_LOCKED = 0.75
CONF_SWEEP  = [0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85]

RADIUS_FRAC = 0.5              # ambang cocok pusat, kelipatan jarak tanam
R_GRAPH     = 1.5              # radius graf kontak, kelipatan jarak tanam
DEG_LAYER2  = 5.74             # pembanding Eg9PP, pohon bagian dalam

import y12                                  # noqa: E402
import centre_eval_folds as cef             # noqa: E402

TAG = "%s_%s_%s" % (os.path.splitext(MODEL)[0], ARM, TAG_SUFFIX)
print("tag hasil : %s   -> yolo12_results/%s.json" % (TAG, TAG))
print("lipatan   : %s" % FOLD)
print("setelan   : %s, %d epoch, imgsz %d, seed %d" % (MODEL, EPOCHS, IMGSZ, SEED))


tag hasil : yolo12n_base_1fold   -> yolo12_results/yolo12n_base_1fold.json
lipatan   : fold0
setelan   : yolo12n.pt, 30 epoch, imgsz 640, seed 42


## 1 · Preflight — gagal sekarang, bukan 20 menit lagi

Menjalankan satu matmul sungguhan di GPU. Build torch yang salah (mis. arsitektur
kartu sudah dibuang dari versi CUDA-nya) baru gagal saat kernel pertama jalan,
bukan saat impor — jadi lebih baik dipicu di sini.


In [2]:
import torch

print("torch        %s   | cuda build %s" % (torch.__version__, torch.version.cuda))
import ultralytics; print("ultralytics  %s" % ultralytics.__version__)

if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("gpu          %s (sm_%d%d, %.1f GB)"
          % (torch.cuda.get_device_name(0), cc[0], cc[1], vram))
    x = torch.randn(512, 512, device="cuda")
    float((x @ x).sum()); torch.cuda.synchronize()
    print("kernel       OK")
    print("\nperkiraan waktu: ~15-20 menit untuk %d epoch pada satu lipatan" % EPOCHS)
    if vram < 5.5 and BATCH is None:
        print("CATATAN: VRAM < 5,5 GB -> setel BATCH = 8 di sel setelan bila OOM.")
else:
    print("gpu          TIDAK ADA -> latih di CPU")
    print("\nPERINGATAN: %d epoch di CPU memakan BERJAM-JAM. Untuk sekadar memeriksa" % EPOCHS)
    print("pipeline berjalan, turunkan EPOCHS ke 2-3 - tetapi angkanya lalu tidak")
    print("sebanding dengan apa pun dan JANGAN dilaporkan.")


torch        2.13.0+cu130   | cuda build 13.0
ultralytics  8.4.96
gpu          NVIDIA GeForce RTX 5060 Laptop GPU (sm_120, 8.0 GB)
kernel       OK

perkiraan waktu: ~15-20 menit untuk 30 epoch pada satu lipatan


## 2 · Data dan harness

`y12.build()` menghubungkan (hardlink) ubin `ds_B` ke `yolo12/` dan menulis label
YOLO plus satu yaml per lipatan. Ia **tidak menyediakan** split acak: hanya ada 3
ortomosaik dan ubin Roboflow bertindih ~30×, jadi split acak bocor 100%
(`../layer1_data_audit/AUDIT_REPORT.md`).

`extra_mode="ignore"` — ubin dari sumber luar tidak ikut, sama seperti lari naskah.


In [3]:
t0 = time.time()
folds = y12.build(extra_mode="ignore")
print("\nsiap dalam %.0f dtk" % (time.time() - t0))

assert FOLD in folds, "%s tidak ada. Tersedia: %s" % (FOLD, folds)
held = cef.ortho_of(FOLD)
print("\nlipatan %s menahan ortomosaik %s" % (FOLD, held))

for r in y12.class_balance([FOLD]):
    print("  val: %d citra | Healthy %d kotak | Unhealthy %d kotak (%.2f%%)"
          % (r["images"], r["healthy"], r["unhealthy"], r["pct_unhealthy"]))

gt_xy, gt_lab = y12.gt_trees(held)
print("  POHON UNIK di ortomosaik ini: %d (positif Unhealthy: %d)"
      % (len(gt_xy), int((gt_lab == 1).sum())))
print("\n  Unit analisisnya POHON UNIK, bukan kotak anotasi. Jangan pernah")
print("  mengutip jumlah kotak sebagai ukuran sampel (larangan #2 README).")


citra   : 2303  kotak: 151060  (dataset B)
  fold0  tahan 44000_16000  train= 1566  val= 737
  fold1  tahan 44000_4000   train= 1536  val= 767
  fold2  tahan 52000_20000  train= 1504  val= 799

siap dalam 2 dtk

lipatan fold0 menahan ortomosaik 44000_16000
  val: 737 citra | Healthy 44282 kotak | Unhealthy 523 kotak (1.17%)
  POHON UNIK di ortomosaik ini: 1379 (positif Unhealthy: 17)

  Unit analisisnya POHON UNIK, bukan kotak anotasi. Jangan pernah
  mengutip jumlah kotak sebagai ukuran sampel (larangan #2 README).


## 3 · Melatih

Satu panggilan ke `y12.train_arm()` — fungsi yang sama yang menghasilkan angka
3 lipatan. `tag_suffix="1fold"` memisahkan ruang namanya, jadi:

* hasil ditulis ke `yolo12_results/yolo12n_base_1fold.json`, **bukan** ke
  `yolo12n_base.json`;
* bobot mendarat di `yolo12_runs/yolo12n_base_1fold_<fold>_s42/`, **bukan** menimpa
  bobot demo di `yolo12n_base_<fold>_s42/`.

Sel ini juga **dapat dilanjutkan**: kalau kernel mati, jalankan ulang — lari yang
sudah selesai dilewati, bukan diulang.


In [4]:
t0 = time.time()
out = y12.train_arm(ARM, model=MODEL, folds=[FOLD], seeds=(SEED,),
                    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
                    cache=CACHE, tag_suffix=TAG_SUFFIX, resume_ok=True, verbose=True)
mins = (time.time() - t0) / 60
print("\nselesai dalam %.1f menit" % mins)

det = out["runs"]["%s|%d" % (FOLD, SEED)]
print("\n== metrik deteksi (SEKUNDER - berlangit-langit label) ==")
print("  mAP50     %.4f" % det["map50"])
print("  mAP50-95  %.4f" % det["map"])
for k, v in sorted(det["ap50"].items()):
    print("  AP50 %-10s %.4f" % (k, v))
print("\n  mAP di sini adalah RATA-RATA atas dua kelas, dan kelas Unhealthy hanya")
print("  bersandar pada belasan pohon unik. Angka ini BUKAN kemampuan deteksi")
print("  tajuk; metrik utamanya ada di bagian berikutnya.")


New https://pypi.org/project/ultralytics/8.4.116 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.96  Python-3.13.14 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\Brian\Desktop\Project\Datathon 2026\Polished AI Concept Paper\oil-palm-detection-datathon-2026\layer1_build\yolo12\fold0.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=N

## 4 · Metrik UTAMA — pusat tajuk pada pohon unik

Kotak kebenaran-dasar `ds_B` adalah **cap berukuran tetap** yang ditempelkan pada
pusat tajuk, bukan kotak yang digambar per tajuk (`LABEL_QUALITY_AUDIT.md`).
Karena itu mAP punya langit-langit yang **tidak bergantung model**, dan yang
dikonsumsi Lapisan 2 adalah **koordinat**, bukan kotak.

Evaluasinya dilakukan pada **pohon unik** setelah deteksi dari ubin bertindih
digabungkan — bukan per ubin, yang akan merata-ratakan ~32 tampilan berkorelasi
dari pohon yang sama.

### Kenapa ambangnya tidak dipilih di sini

Di lari 3 lipatan, ambang `conf` untuk lipatan *f* dipilih dari kurva F1 lipatan
**lain** saja, supaya ortomosaik yang ditahan tidak ikut memilih apa pun tentang
dirinya sendiri. Ketiganya mendarat di **0,75**.

Dengan **satu** lipatan prosedur itu mustahil — tidak ada lipatan lain untuk
memilih darinya. Maka ambangnya **diwarisi terkunci pada 0,75**, bukan dipilih di
sini. Memilih ambang dengan melihat kurva lipatan yang ditahan adalah memilih
hiperparameter di atas data uji, dan itu justru bentuk kebocoran yang dijaga
seluruh harness ini.

Sapuan di bawah dicetak untuk **melihat kepekaannya**, bukan untuk memilih.


In [5]:
sp = cef.spacing_of(gt_xy)
print("jarak tanam %s: %.1f px | radius cocok %.1f px (%.2f x jarak tanam)"
      % (held, sp, RADIUS_FRAC * sp, RADIUS_FRAC))

w = cef.weights_of(FOLD, tag=TAG)
assert w, "bobot tidak ditemukan untuk %s - sel latih belum selesai?" % TAG
print("bobot: %s\n" % os.path.relpath(w, BASE))

curve = {}
print("%-6s %8s %8s %8s %8s %10s" % ("conf", "n_pred", "P", "R", "F1", "RMSE/jarak"))
for c in CONF_SWEEP:
    p_xy, _, _ = y12.predict_global(w, FOLD, conf=c, imgsz=IMGSZ, verbose=False)
    s = cef.score(p_xy, gt_xy, sp)
    s["pred_xy"] = p_xy
    curve[c] = s
    mark = "  <- TERKUNCI" if abs(c - CONF_LOCKED) < 1e-9 else ""
    print("%-6.2f %8d %8.3f %8.3f %8.3f %10.3f%s"
          % (c, s["n_pred"], s["precision"], s["recall"], s["f1"],
             s["rmse_frac_spacing"], mark))


jarak tanam 44000_16000: 105.8 px | radius cocok 52.9 px (0.50 x jarak tanam)
bobot: yolo12_runs\yolo12n_base_1fold_fold0_s42\weights\best.pt

conf     n_pred        P        R       F1 RMSE/jarak
0.25       1595    0.864    0.999    0.927      0.051
0.35       1526    0.900    0.996    0.946      0.050
0.45       1494    0.919    0.996    0.956      0.050
0.55       1465    0.936    0.994    0.964      0.050
0.65       1433    0.956    0.993    0.974      0.050
0.75       1409    0.970    0.991    0.981      0.050  <- TERKUNCI
0.85       1374    0.989    0.985    0.987      0.050


In [6]:
best_here = max(CONF_SWEEP, key=lambda c: curve[c]["f1"])
S = curve[CONF_LOCKED]

print("== METRIK UTAMA TAHAP 1 (conf terkunci %.2f, %s, %s) ==" % (CONF_LOCKED, FOLD, held))
print("  Presisi   %.4f" % S["precision"])
print("  Recall    %.4f" % S["recall"])
print("  F1        %.4f   <- angka utamanya" % S["f1"])
print("  RMSE      %.1f px  = %.3f x jarak tanam" % (S["rmse_px"], S["rmse_frac_spacing"]))
print("  TP %d | FP %d | FN %d  (dari %d pohon unik)"
      % (S["tp"], S["fp"], S["fn"], S["n_gt"]))

if abs(best_here - CONF_LOCKED) > 1e-9:
    print("\n  CATATAN: pada lipatan INI, F1 tertinggi ada di conf=%.2f (%.4f), bukan %.2f"
          % (best_here, curve[best_here]["f1"], CONF_LOCKED))
    print("  Selisihnya %+.4f. Angka itu TIDAK dipakai: memilihnya berarti menyetel"
          % (curve[best_here]["f1"] - S["f1"]))
    print("  ambang di atas ortomosaik uji. 0,75 tetap dipertahankan.")
else:
    print("\n  Ambang terkunci kebetulan juga optimum di lipatan ini.")

print("\n  Kurva F1 yang DATAR terhadap radius pencocokan = pusatnya memang tepat;")
print("  kurva yang menanjak tajam = 'benar' hanya karena ambangnya longgar.")
for r in y12.radius_sweep(w, FOLD, fracs=(0.25, 0.35, 0.5, 0.75, 1.0),
                          conf=CONF_LOCKED, imgsz=IMGSZ):
    print("    radius %.2f x jarak tanam (%5.1f px): F1 %.4f" % (r["frac"], r["radius_px"], r["f1"]))


== METRIK UTAMA TAHAP 1 (conf terkunci 0.75, fold0, 44000_16000) ==
  Presisi   0.9702
  Recall    0.9913
  F1        0.9806   <- angka utamanya
  RMSE      5.3 px  = 0.050 x jarak tanam
  TP 1367 | FP 42 | FN 12  (dari 1379 pohon unik)

  CATATAN: pada lipatan INI, F1 tertinggi ada di conf=0.85 (0.9873), bukan 0.75
  Selisihnya +0.0067. Angka itu TIDAK dipakai: memilihnya berarti menyetel
  ambang di atas ortomosaik uji. 0,75 tetap dipertahankan.

  Kurva F1 yang DATAR terhadap radius pencocokan = pusatnya memang tepat;
  kurva yang menanjak tajam = 'benar' hanya karena ambangnya longgar.
    radius 0.25 x jarak tanam ( 26.5 px): F1 0.9806
    radius 0.35 x jarak tanam ( 37.0 px): F1 0.9806
    radius 0.50 x jarak tanam ( 52.9 px): F1 0.9806
    radius 0.75 x jarak tanam ( 79.4 px): F1 0.9806
    radius 1.00 x jarak tanam (105.8 px): F1 0.9806


## 5 · Angka uji jembatan — derajat graf kontak

Satu-satunya besaran yang menghubungkan kedua lapisan tanpa menggabungkan
datanya: **derajat rata-rata pada r = 1,5 × jarak tanam, pohon bagian dalam saja.**

Pohon di tepi kehilangan tetangga yang terpotong bingkai citra; memasukkannya
akan menyeret derajat ke bawah karena alasan yang tidak ada hubungannya dengan
jarak tanam. Hanya kolom **dalam** yang sebanding dengan 5,74 milik Eg9PP.


In [7]:
d_gt = cef.degree(gt_xy, R_GRAPH * sp)
d_pr = cef.degree(S["pred_xy"], R_GRAPH * sp)

print("derajat @ r = %.1f x jarak tanam  (%s)" % (R_GRAPH, held))
print("  kotak kebenaran-dasar : semua %.3f | DALAM %.3f  (n=%d)" % (d_gt[0], d_gt[1], d_gt[2]))
print("  prediksi detektor     : semua %.3f | DALAM %.3f  (n=%d)" % (d_pr[0], d_pr[1], d_pr[2]))
print("  ongkos memakai detektor: %+.3f (%+.1f%%)"
      % (d_pr[1] - d_gt[1], 100 * (d_pr[1] - d_gt[1]) / d_gt[1]))
print("\n  Lapisan 2 (Eg9PP, pohon dalam) = %.2f" % DEG_LAYER2)
print("  selisih lipatan ini terhadap Eg9PP: %+.3f (%+.1f%%)"
      % (d_pr[1] - DEG_LAYER2, 100 * (d_pr[1] - DEG_LAYER2) / DEG_LAYER2))
print("\n  Ini SATU lipatan. Angka naskah 5,54 +/- 0,12 adalah rata-rata 3 ortomosaik;")
print("  jangan menggantikannya dengan angka di atas.")


derajat @ r = 1.5 x jarak tanam  (44000_16000)
  kotak kebenaran-dasar : semua 5.524 | DALAM 5.682  (n=1242)
  prediksi detektor     : semua 5.479 | DALAM 5.614  (n=1297)
  ongkos memakai detektor: -0.067 (-1.2%)

  Lapisan 2 (Eg9PP, pohon dalam) = 5.74
  selisih lipatan ini terhadap Eg9PP: -0.126 (-2.2%)

  Ini SATU lipatan. Angka naskah 5,54 +/- 0,12 adalah rata-rata 3 ortomosaik;
  jangan menggantikannya dengan angka di atas.


## 6 · Perbandingan dengan lari 3 lipatan yang tercatat

Dua perbandingan di bawah, dan hanya **satu** di antaranya sah.

* **Lipatan yang sama lawan lipatan yang sama** — sah. Ia menjawab "apakah mesin
  dan versi pustaka ini menghasilkan angka yang sama?"
* **Lipatan ini lawan rata-rata 3 lipatan** — **tidak sah**, dicetak hanya sebagai
  konteks dan ditandai. Satu titik bukan rata-rata.


In [8]:
ref_p = os.path.join(y12.RESDIR, "yolo12n_base.json")
cen_p = os.path.join(y12.RESDIR, "centre_eval.json")

if os.path.isfile(ref_p):
    ref = json.load(open(ref_p))["runs"].get("%s|%d" % (FOLD, SEED))
    if ref:
        print("== deteksi, LIPATAN YANG SAMA (sah) ==")
        print("  %-10s %10s %10s %10s" % ("", "sekarang", "tercatat", "selisih"))
        for k, lab in (("map50", "mAP50"), ("map", "mAP50-95")):
            print("  %-10s %10.4f %10.4f %+10.4f" % (lab, det[k], ref[k], det[k] - ref[k]))

if os.path.isfile(cen_p):
    cen = json.load(open(cen_p))
    r = cen["folds"].get(FOLD)
    if r:
        print("\n== pusat tajuk, LIPATAN YANG SAMA (sah) ==")
        print("  %-10s %10s %10s %10s" % ("", "sekarang", "tercatat", "selisih"))
        for k, lab in (("precision", "Presisi"), ("recall", "Recall"), ("f1", "F1")):
            print("  %-10s %10.4f %10.4f %+10.4f" % (lab, S[k], r[k], S[k] - r[k]))
        print("  %-10s %10.3f %10.3f %+10.3f"
              % ("deg dalam", d_pr[1], r["deg_pred_inner"], d_pr[1] - r["deg_pred_inner"]))

    f1s = [v["f1"] for v in cen["folds"].values()]
    if len(f1s) > 1:
        import statistics as st
        print("\n== KONTEKS SAJA - JANGAN DIADU ==")
        print("  rata-rata 3 ortomosaik : F1 %.3f +/- %.3f  (n=3, mean +/- std)"
              % (st.mean(f1s), st.stdev(f1s)))
        print("  lipatan ini            : F1 %.3f          (n=1, TANPA std)" % S["f1"])
        print("\n  Satu titik lawan sebuah rata-rata. `paired()` tidak berlaku,")
        print("  dan menuliskan keduanya berdampingan di naskah adalah salah kutip")
        print("  (00_ANGKA_FINAL.md bagian E).")


== deteksi, LIPATAN YANG SAMA (sah) ==
               sekarang   tercatat    selisih
  mAP50          0.7018     0.7258    -0.0240
  mAP50-95       0.4837     0.4829    +0.0007

== pusat tajuk, LIPATAN YANG SAMA (sah) ==
               sekarang   tercatat    selisih
  Presisi        0.9702     0.9493    +0.0209
  Recall         0.9913     0.9906    +0.0007
  F1             0.9806     0.9695    +0.0111
  deg dalam       5.614      5.663     -0.049

== KONTEKS SAJA - JANGAN DIADU ==
  rata-rata 3 ortomosaik : F1 0.960 +/- 0.024  (n=3, mean +/- std)
  lipatan ini            : F1 0.981          (n=1, TANPA std)

  Satu titik lawan sebuah rata-rata. `paired()` tidak berlaku,
  dan menuliskan keduanya berdampingan di naskah adalah salah kutip
  (00_ANGKA_FINAL.md bagian E).


## 7 · Simpan — dengan batasnya ikut tertulis

Ringkasan ditulis ke `yolo12_results/centre_eval_1fold.json`. Berkas itu memuat
field `batas` yang menyatakan sendiri apa yang tidak boleh dilakukan padanya,
supaya siapa pun yang membacanya nanti tidak perlu menebak.


In [9]:
summary = dict(
    tag=TAG, fold=FOLD, ortho=held, seed=SEED,
    setting=dict(model=MODEL, epochs=EPOCHS, imgsz=IMGSZ, arm=ARM,
                 batch=BATCH, cache=CACHE, workers=WORKERS),
    n_folds=1, n_seeds=1,
    conf=CONF_LOCKED,
    conf_selection=("DIWARISI dari lari 3 lipatan (yang memilihnya silang-lipatan). "
                    "TIDAK dipilih di sini: dengan satu lipatan, memilih ambang "
                    "berarti menyetelnya di atas ortomosaik uji."),
    detection={k: det[k] for k in ("map50", "map", "mp", "mr")},
    ap50=det["ap50"],
    centre={k: S[k] for k in ("n_pred", "n_gt", "tp", "fp", "fn",
                              "precision", "recall", "f1", "rmse_px",
                              "rmse_frac_spacing")},
    spacing_px=sp,
    degree={"r_graph": R_GRAPH,
            "gt_all": d_gt[0], "gt_inner": d_gt[1], "n_inner_gt": d_gt[2],
            "pred_all": d_pr[0], "pred_inner": d_pr[1], "n_inner_pred": d_pr[2],
            "layer2_inner": DEG_LAYER2},
    conf_curve={("%.2f" % c): {k: v[k] for k in ("n_pred", "precision", "recall",
                                                 "f1", "rmse_frac_spacing")}
                for c, v in curve.items()},
    batas=[
        "1 lipatan, 1 seed - TIDAK ada mean +/- std.",
        "paired() TIDAK berlaku: tidak ada pasangan (lipatan, seed).",
        "JANGAN diadu dengan F1 pusat 0,960 +/- 0,024 milik lari 3 lipatan.",
        "JANGAN masuk 00_RINGKASAN.csv maupun 00_ANGKA_FINAL.md.",
        "Bukan replikasi. Ini lari verifikasi pipeline di satu mesin.",
        "Perbandingan yang sah hanya lipatan-lawan-lipatan-yang-sama.",
    ],
)

p = os.path.join(y12.RESDIR, "centre_eval_1fold.json")
json.dump(summary, open(p, "w"), indent=2)
print("ditulis: %s" % os.path.relpath(p, BASE))
print("\n" + "=" * 70)
print("RINGKASAN  %s / %s  (1 lipatan, 1 seed)" % (FOLD, held))
print("=" * 70)
print("  F1 pusat tajuk   %.4f   <- metrik utama Tahap 1" % S["f1"])
print("  Presisi/Recall   %.4f / %.4f" % (S["precision"], S["recall"]))
print("  RMSE pusat       %.3f x jarak tanam" % S["rmse_frac_spacing"])
print("  mAP50            %.4f   (sekunder, berlangit-langit)" % det["map50"])
print("  derajat dalam    %.3f   (Eg9PP = %.2f)" % (d_pr[1], DEG_LAYER2))
print("\n  Klaim maksimum: pipeline Tahap 1 berjalan ujung ke ujung di mesin ini")
print("  dan lipatan %s mendarat di dekat angka yang tercatat untuk lipatan yang" % FOLD)
print("  sama. BUKAN angka naskah, BUKAN replikasi.")


ditulis: yolo12_results\centre_eval_1fold.json

RINGKASAN  fold0 / 44000_16000  (1 lipatan, 1 seed)
  F1 pusat tajuk   0.9806   <- metrik utama Tahap 1
  Presisi/Recall   0.9702 / 0.9913
  RMSE pusat       0.050 x jarak tanam
  mAP50            0.7018   (sekunder, berlangit-langit)
  derajat dalam    5.614   (Eg9PP = 5.74)

  Klaim maksimum: pipeline Tahap 1 berjalan ujung ke ujung di mesin ini
  dan lipatan fold0 mendarat di dekat angka yang tercatat untuk lipatan yang
  sama. BUKAN angka naskah, BUKAN replikasi.
